In [2]:
# ===========================================================================
# LERN-NOTEBOOK: Threading & Multiprocessing in Python
# ===========================================================================
# Dieses Notebook erklärt Threading und Multiprocessing anhand von
# einfachen, sichtbaren Beispielen — mit Ausgaben die zeigen WAS passiert.
# Jede Zelle baut auf der vorherigen auf.
# ===========================================================================


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  KAPITEL 0 — Imports                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import threading
import multiprocessing
import time
import os
from queue import Queue

### 🧵 Threading & Multiprocessing — Lern-Notebook

#### Was lernst du hier?

| Kapitel | Thema |
|---------|-------|
| 1 | Was ist ein Thread? Sichtbar machen |
| 2 | Threads parallel starten |
| 3 | Das Race-Condition Problem |
| 4 | Lock — der "Türsteher" |
| 5 | Die Queue — sicherer Datenaustausch |
| 6 | Daemon-Threads |
| 7 | Wann Thread, wann Prozess? |
| 8 | Multiprocessing — echte Parallelität |
| 9 | Unser IFS-Scanner — Prinzip erklärt |

### KAPITEL 1 — Was ist ein Thread?
Stell dir vor, du kochst:
- **Ohne Threading**: erst Wasser kochen → warten → dann Gemüse schneiden
- **Mit Threading**:  Wasser kochen UND gleichzeitig Gemüse schneiden

Ein **Thread** ist ein eigenständiger Ausführungsstrang innerhalb deines Programms. Mehrere Threads teilen sich denselben Speicher (das ist Stärke UND Gefahr — dazu später mehr).

Zunächst: ein einzelner Thread, sichtbar gemacht.

In [3]:
def aufgabe(name: str, dauer: float):
    """Eine einfache Aufgabe die etwas Zeit braucht."""
    print(f"  [{name}] startet   — Zeit: {time.strftime('%H:%M:%S')}")
    time.sleep(dauer)   # simuliert Arbeit (z.B. Netzwerk-Warten)
    print(f"  [{name}] fertig    — Zeit: {time.strftime('%H:%M:%S')}")

print("=== Ohne Threading (nacheinander) ===")
t0 = time.time()

aufgabe("Aufgabe-A", 2)
aufgabe("Aufgabe-B", 2)

print(f"Gesamtzeit: {time.time()-t0:.1f}s  (erwartet: ~4s)\n")

=== Ohne Threading (nacheinander) ===
  [Aufgabe-A] startet   — Zeit: 23:22:38
  [Aufgabe-A] fertig    — Zeit: 23:22:40
  [Aufgabe-B] startet   — Zeit: 23:22:40
  [Aufgabe-B] fertig    — Zeit: 23:22:42
Gesamtzeit: 4.0s  (erwartet: ~4s)



### KAPITEL 2 — Threads parallel starten
Jetzt starten wir BEIDE Aufgaben gleichzeitig. Beachte die Ausgabe: die Zeilen beider Threads erscheinen durchgemischt — das zeigt echte Parallelität!
**Wichtige Methoden:**
- `t = threading.Thread(target=funktion, args=(arg1, arg2))`
   → erstellt einen Thread, startet ihn aber noch NICHT
- `t.start()`
  → Thread startet jetzt
- `t.join()`
  → Hauptprogramm WARTET bis dieser Thread fertig ist

In [4]:
print("=== Mit Threading (gleichzeitig) ===")
t0 = time.time()

# Threads erstellen
t_a = threading.Thread(target=aufgabe, args=("Aufgabe-A", 2))
t_b = threading.Thread(target=aufgabe, args=("Aufgabe-B", 2))

# Threads starten
t_a.start()
t_b.start()

# Warten bis beide fertig sind
t_a.join()
t_b.join()

print(f"Gesamtzeit: {time.time()-t0:.1f}s  (erwartet: ~2s — halb so lang!)\n")

=== Mit Threading (gleichzeitig) ===
  [Aufgabe-A] startet   — Zeit: 23:22:54
  [Aufgabe-B] startet   — Zeit: 23:22:54
  [Aufgabe-A] fertig    — Zeit: 23:22:56
  [Aufgabe-B] fertig    — Zeit: 23:22:56
Gesamtzeit: 2.0s  (erwartet: ~2s — halb so lang!)



### Kapitel 3 — Das Race-Condition Problem ⚠️

Threads teilen sich den Speicher. Das klingt praktisch — ist aber gefährlich!

**Beispiel:** Zwei Threads erhöhen einen gemeinsamen Zähler.
Jeder soll ihn 100.000 Mal um 1 erhöhen → Erwartung: 200.000

Was intern passiert (vereinfacht):
```
Thread A liest:  zaehler = 5
Thread B liest:  zaehler = 5   ← liest denselben Wert!
Thread A schreibt: zaehler = 6
Thread B schreibt: zaehler = 6  ← überschreibt A's Arbeit!
```
Ergebnis: eine Erhöhung geht verloren → **Race Condition**


In [6]:
zaehler = 0

def erhoehe_unsicher(n: int):
    global zaehler
    for _ in range(n):
        zaehler += 1    # NICHT thread-sicher!

zaehler = 0
t1 = threading.Thread(target=erhoehe_unsicher, args=(100_000,))
t2 = threading.Thread(target=erhoehe_unsicher, args=(100_000,))

t1.start(); t2.start()
t1.join();  t2.join()

print(f"=== Race Condition ===")
print(f"Erwartet : 200.000")
print(f"Erhalten : {zaehler:,}  ← wahrscheinlich WENIGER (nicht deterministisch!)\n")

=== Race Condition ===
Erwartet : 200.000
Erhalten : 200,000  ← wahrscheinlich WENIGER (nicht deterministisch!)



## Kapitel 4 — Lock: der "Türsteher" 🔒

Ein **Lock** ist wie ein Türsteher vor einem Raum:
- Nur EIN Thread darf gleichzeitig rein
- Alle anderen warten draußen
- Wenn der Thread fertig ist, gibt er den Lock frei

```
lock = threading.Lock()

with lock:          # ← Lock holen (warten falls belegt)
    zaehler += 1    # ← kritischer Bereich: nur ein Thread gleichzeitig
                    # ← Lock automatisch freigeben beim Verlassen des with-Blocks
```

**Nachteil:** Der Lock erzeugt etwas Wartezeit — aber das Ergebnis ist korrekt!


In [6]:
lock    = threading.Lock()
zaehler = 0

def erhoehe_sicher(n: int):
    global zaehler
    for _ in range(n):
        with lock:          # nur ein Thread gleichzeitig
            zaehler += 1

zaehler = 0
t1 = threading.Thread(target=erhoehe_sicher, args=(100_000,))
t2 = threading.Thread(target=erhoehe_sicher, args=(100_000,))

t1.start(); t2.start()
t1.join();  t2.join()

print(f"=== Mit Lock ===")
print(f"Erwartet : 200.000")
print(f"Erhalten : {zaehler:,}  ← immer korrekt!\n")

=== Mit Lock ===
Erwartet : 200.000
Erhalten : 200,000  ← immer korrekt!



### Kapitel 5 — Die Queue: sicherer Datenaustausch 📬

Eine **Queue** ist wie ein Briefkasten zwischen Threads:
- **Producer** (Erzeuger) legt Aufgaben rein: `q.put(aufgabe)`
- **Consumer** (Verbraucher) holt Aufgaben raus: `aufgabe = q.get()`
- Die Queue ist von Haus aus **thread-sicher** — kein Lock nötig!

Das **Poison-Pill-Muster**: Um einem Thread zu sagen "du bist fertig",
legt man ein `None` in die Queue. Der Thread prüft darauf und beendet sich.

```
q.put(None)   ← Poison Pill
```

**`q.task_done()`** und **`q.join()`**:
- Nach jeder `q.get()` muss `q.task_done()` gerufen werden
- `q.join()` blockiert bis ALLE tasks_done gerufen wurden
- So weiß das Hauptprogramm: alle Aufgaben sind wirklich erledigt

In [7]:
def produzent(q: Queue, aufgaben: list):
    """Legt Aufgaben in die Queue."""
    for a in aufgaben:
        print(f"  Produzent: lege '{a}' in Queue")
        q.put(a)
        time.sleep(0.1)

def konsument(q: Queue, name: str):
    """Holt Aufgaben aus der Queue und bearbeitet sie."""
    while True:
        aufgabe = q.get()

        if aufgabe is None:             # Poison Pill
            print(f"  [{name}] beendet sich")
            q.task_done()
            break

        print(f"  [{name}] bearbeitet: {aufgabe}")
        time.sleep(0.3)                 # simuliert Arbeit
        q.task_done()

print("=== Queue-Beispiel (1 Produzent, 2 Konsumenten) ===")

q = Queue()
aufgaben = ['Datei-1', 'Datei-2', 'Datei-3', 'Datei-4', 'Datei-5']

# 2 Konsumenten starten
k1 = threading.Thread(target=konsument, args=(q, 'Konsument-1'))
k2 = threading.Thread(target=konsument, args=(q, 'Konsument-2'))
k1.start()
k2.start()

# Produzent läuft im Hauptthread
produzent(q, aufgaben)

# Warten bis alle Aufgaben erledigt
q.join()

# Konsumenten beenden
q.put(None)
q.put(None)
k1.join()
k2.join()

print("Alle Aufgaben erledigt!\n")


=== Queue-Beispiel (1 Produzent, 2 Konsumenten) ===
  Produzent: lege 'Datei-1' in Queue
  [Konsument-1] bearbeitet: Datei-1
  Produzent: lege 'Datei-2' in Queue
  [Konsument-2] bearbeitet: Datei-2
  Produzent: lege 'Datei-3' in Queue
  [Konsument-1] bearbeitet: Datei-3
  Produzent: lege 'Datei-4' in Queue
  [Konsument-2] bearbeitet: Datei-4
  Produzent: lege 'Datei-5' in Queue
  [Konsument-1] bearbeitet: Datei-5
  [Konsument-2] beendet sich
  [Konsument-1] beendet sich
Alle Aufgaben erledigt!



## Kapitel 6 — Daemon-Threads 👻

Ein **normaler Thread** hält das Programm am Leben bis er fertig ist.
Ein **Daemon-Thread** wird automatisch beendet wenn das Hauptprogramm endet —
egal ob er fertig ist oder nicht.

In unserem IFS-Scanner setzen wir `daemon=True`:
```Python
   t = threading.Thread(target=..., daemon=True)
```

**Warum?** Wenn ein Fehler passiert und das Hauptprogramm abbricht, sollen die Worker-Threads nicht ewig weiterlaufen. Wir steuern das Ende sauber über Poison Pills + `q.join()` — der Daemon-Status ist nur das Sicherheitsnetz.


In [8]:
def hintergrund_job():
    for i in range(10):
        print(f"  Daemon läuft noch... ({i+1}/10)")
        time.sleep(0.3)

print("=== Daemon-Thread ===")
d = threading.Thread(target=hintergrund_job, daemon=True)
d.start()

time.sleep(0.8)   # Hauptprogramm macht etwas
print("Hauptprogramm fertig — Daemon wird jetzt abgebrochen\n")
# d wird automatisch beendet, wir rufen kein d.join()

=== Daemon-Thread ===
  Daemon läuft noch... (1/10)
  Daemon läuft noch... (2/10)
  Daemon läuft noch... (3/10)
Hauptprogramm fertig — Daemon wird jetzt abgebrochen



  Daemon läuft noch... (4/10)
  Daemon läuft noch... (5/10)
  Daemon läuft noch... (6/10)
  Daemon läuft noch... (7/10)
  Daemon läuft noch... (8/10)
  Daemon läuft noch... (9/10)
  Daemon läuft noch... (10/10)


### Kapitel 7 — Wann Thread, wann Prozess? 🤔

### Der GIL (Global Interpreter Lock)

Python hat eine interne Sperre: den **GIL**.
Er erlaubt immer nur EINEM Thread gleichzeitig Python-Code auszuführen.

Das klingt nach "Threading ist nutzlos" — aber:

| Aufgabe | Threads | Prozesse |
|---------|---------|----------|
| Netzwerk-I/O (warten auf Server) | ✅ ideal | möglich |
| Datei-I/O (warten auf Festplatte) | ✅ ideal | möglich |
| Reine CPU-Arbeit (Berechnungen) | ❌ kein Gewinn | ✅ ideal |

**Warum Threads bei I/O trotzdem schnell sind:**
Während ein Thread WARTET (auf Netzwerk, Festplatte), gibt er den GIL frei.
Ein anderer Thread kann dann arbeiten.
→ Bei unserem IFS-Scanner warten die Threads fast die ganze Zeit
  auf `os.scandir()` → Threads sind genau richtig!

**Prozesse umgehen den GIL** — jeder Prozess hat seinen eigenen
Python-Interpreter. Für CPU-intensive Arbeit (z.B. Bildverarbeitung,
 mathematische Berechnungen) sind Prozesse besser.


In [10]:
print("=== GIL-Demonstration ===")
print()

def cpu_arbeit(n: int):
    """Reine CPU-Arbeit: sehr viele Berechnungen."""
    summe = 0
    for i in range(n):
        summe += i * i
    return summe

def io_arbeit(n: int):
    """I/O-Simulation: warten wie beim Netzwerk."""
    for _ in range(n):
        time.sleep(0.01)

N = 5_000_000

# CPU-Arbeit: Threads vs. seriell
print("CPU-Arbeit (Berechnungen):")
t0 = time.time()
cpu_arbeit(N)
cpu_arbeit(N)
t_seriell = time.time() - t0

t0 = time.time()
t1 = threading.Thread(target=cpu_arbeit, args=(N,))
t2 = threading.Thread(target=cpu_arbeit, args=(N,))
t1.start(); t2.start()
t1.join();  t2.join()
t_threads = time.time() - t0

print(f"  Seriell : {t_seriell:.2f}s")
print(f"  Threads : {t_threads:.2f}s  ← kaum schneller (GIL!)")
print()

# I/O-Arbeit: Threads vs. seriell
N_IO = 10
print("I/O-Arbeit (Warten simuliert):")
t0 = time.time()
io_arbeit(N_IO)
io_arbeit(N_IO)
t_seriell = time.time() - t0

t0 = time.time()
t1 = threading.Thread(target=io_arbeit, args=(N_IO,))
t2 = threading.Thread(target=io_arbeit, args=(N_IO,))
t1.start(); t2.start()
t1.join();  t2.join()
t_threads = time.time() - t0

print(f"  Seriell : {t_seriell:.2f}s")
print(f"  Threads : {t_threads:.2f}s  ← ~2x schneller (GIL freigegeben beim Warten!)\n")

=== GIL-Demonstration ===

CPU-Arbeit (Berechnungen):
  Seriell : 0.73s
  Threads : 0.73s  ← kaum schneller (GIL!)

I/O-Arbeit (Warten simuliert):
  Seriell : 0.21s
  Threads : 0.11s  ← ~2x schneller (GIL freigegeben beim Warten!)



### Kapitel 8 — Multiprocessing: echte Parallelität 🖥️🖥️

Multiprocessing startet **echte separate Prozesse** — jeder mit eigenem
Python-Interpreter und eigenem Speicher. Kein GIL!

**Wichtiger Unterschied zu Threads:**

| | Threads | Prozesse |
|-|---------|----------|
| Speicher | geteilt (schnell, aber gefährlich) | getrennt (sicher, aber Overhead) |
| Kommunikation | direkt über Variablen | nur über Queue/Pipe (serialisiert) |
| Start-Overhead | sehr gering (~ms) | spürbar (~0.5-2s auf Windows) |
| GIL | vorhanden | keiner (jeder Prozess hat eigenen) |

### Warum brauchen wir ein Worker-.py-File?

Wenn ein neuer Prozess gestartet wird, muss Python die Worker-Funktion
**importieren** können. In einem Notebook sind Funktionen nicht importierbar —
sie leben nur in der Zelle. Daher: Worker-Funktionen in eine .py-Datei!

Außerdem: `if __name__ == '__main__'` schützt vor dem ungewollten
erneuten Ausführen des gesamten Codes beim Prozess-Start (Windows!).

In [11]:
print("=== Multiprocessing — Konzept ===")
print()
print("  In einer .py-Datei würde das so aussehen:")
print()
print("  from concurrent.futures import ProcessPoolExecutor")
print()
print("  with ProcessPoolExecutor(max_workers=4) as pool:")
print("      futures = [pool.submit(quadriere, 1_000_000) for _ in range(4)]")
print("      ergebnisse = [f.result() for f in futures]")
print()
print("  → Jedes pool.submit() startet einen echten Prozess auf einem CPU-Kern")
print("  → Alle 4 Prozesse rechnen WIRKLICH gleichzeitig (kein GIL)")
print()
print("  Für Threads funktioniert dasselbe mit ThreadPoolExecutor —")
print("  der Code ist identisch, nur der Pool-Typ unterscheidet sich!\n")

=== Multiprocessing — Konzept ===

  In einer .py-Datei würde das so aussehen:

  from concurrent.futures import ProcessPoolExecutor

  with ProcessPoolExecutor(max_workers=4) as pool:
      futures = [pool.submit(quadriere, 1_000_000) for _ in range(4)]
      ergebnisse = [f.result() for f in futures]

  → Jedes pool.submit() startet einen echten Prozess auf einem CPU-Kern
  → Alle 4 Prozesse rechnen WIRKLICH gleichzeitig (kein GIL)

  Für Threads funktioniert dasselbe mit ThreadPoolExecutor —
  der Code ist identisch, nur der Pool-Typ unterscheidet sich!



In [19]:
%%time
from concurrent.futures import ProcessPoolExecutor
from Learn_Threading_Worker import quadriere

with ProcessPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(quadriere, 1_000_000) for _ in range(4)]
    ergebnisse = [f.result() for f in futures]

print(ergebnisse)

[333332833333500000, 333332833333500000, 333332833333500000, 333332833333500000]
CPU times: total: 15.6 ms
Wall time: 318 ms


### Kapitel 9 — Unser IFS-Scanner: alles zusammen 🗂️

Jetzt siehst du warum der Scanner so aufgebaut ist wie er ist:

```
┌─────────────────────────────────────────────────────┐
│  Hauptprogramm (Notebook)                           │
│                                                     │
│  1. get_root_dirs()  → Liste der Root-Dirs          │
│  2. classify()       → klein / groß                 │
│                                                     │
│  ┌──────────────────┐  ┌──────────────────────────┐ │
│  │ KLEINE DIRS      │  │ GROSSE DIRS              │ │
│  │                  │  │                          │ │
│  │ Eine Queue       │  │ ProcessPoolExecutor      │ │
│  │ 16 Threads       │  │ je Prozess: eine Queue   │ │
│  │                  │  │ je Prozess: 16 Threads   │ │
│  │ Threads finden   │  │                          │ │
│  │ Subdir → sofort  │  │ Kein Level-by-Level!     │ │
│  │ in Queue → kein  │  │ Kontinuierliche Arbeit   │ │
│  │ Warten!          │  │                          │ │
│  └──────────────────┘  └──────────────────────────┘ │
│                                                     │
│  3. pd.DataFrame(all_entries)  → eine CSV           │
└─────────────────────────────────────────────────────┘
```

**Warum Queue statt Level-by-Level?**
```
Level-by-Level:   Ebene1 fertig → PAUSE → Ebene2 → PAUSE → Ebene3 ...
Dynamische Queue: Thread findet Subdir → sofort weiter — KEINE Pausen!
```

**Warum daemon=True für die Threads?**
Sicherheitsnetz: falls etwas schiefläuft, laufen keine Zombie-Threads weiter.
Das saubere Ende läuft über Poison Pills + q.join().

**Warum Lock für results.append()?**
Mehrere Threads schreiben gleichzeitig in die results-Liste →
ohne Lock: Race Condition → mit Lock: immer korrekt.


In [12]:
print("=== Mini-Scanner (Lern-Version) ===")
print("Prinzip des IFS-Scanners auf einem lokalen Ordner:\n")

def mini_scanner(startpfad: str, n_threads: int = 4):
    """
    Vereinfachter Scanner — zeigt das Queue-Prinzip sichtbar.
    """
    q       = Queue()
    results = []
    errors  = []
    lock    = threading.Lock()

    def worker():
        while True:
            pfad = q.get()
            if pfad is None:
                q.task_done()
                break
            try:
                with os.scandir(pfad) as scn:
                    for x in scn:
                        if x.is_symlink():
                            continue
                        if x.is_dir(follow_symlinks=False):
                            q.put(x.path)           # sofort in Queue!
                        elif x.is_file(follow_symlinks=False):
                            with lock:
                                results.append(x.path)
            except OSError as e:
                with lock:
                    errors.append(str(e))
            finally:
                q.task_done()

    q.put(startpfad)

    threads = [threading.Thread(target=worker, daemon=True) for _ in range(n_threads)]
    for t in threads: t.start()

    q.join()

    for _ in threads: q.put(None)
    for t in threads: t.join()

    return results, errors

# Test auf lokalem Verzeichnis
testpfad = os.path.expanduser('~')   # Home-Verzeichnis des Benutzers
print(f"Scanne: {testpfad}")
print(f"Threads: 4\n")

t0 = time.time()
dateien, fehler = mini_scanner(testpfad, n_threads=4)
elapsed = time.time() - t0

print(f"Ergebnis:")
print(f"  Gefundene Dateien : {len(dateien):,}")
print(f"  Fehler            : {len(fehler)}")
print(f"  Laufzeit          : {elapsed:.1f}s")
print(f"\nErste 5 Dateien:")
for d in dateien[:5]:
    print(f"  {d}")


=== Mini-Scanner (Lern-Version) ===
Prinzip des IFS-Scanners auf einem lokalen Ordner:

Scanne: C:\Users\stefa
Threads: 4

Ergebnis:
  Gefundene Dateien : 778,054
  Fehler            : 20
  Laufzeit          : 8.3s

Erste 5 Dateien:
  C:\Users\stefa\.condarc
  C:\Users\stefa\.gitconfig
  C:\Users\stefa\NTUSER.DAT
  C:\Users\stefa\ntuser.dat.LOG1
  C:\Users\stefa\ntuser.dat.LOG2


## Zusammenfassung — die wichtigsten Konzepte

| Konzept | Wofür | Wichtigste Methode |
|---------|-------|-------------------|
| `Thread` | Gleichzeitige I/O-Aufgaben | `.start()`, `.join()` |
| `Lock` | Gemeinsame Variablen schützen | `with lock:` |
| `Queue` | Aufgaben zwischen Threads verteilen | `.put()`, `.get()`, `.task_done()`, `.join()` |
| Poison Pill | Thread sauber beenden | `q.put(None)` |
| `daemon=True` | Sicherheitsnetz bei Programmabbruch | `Thread(..., daemon=True)` |
| `ProcessPoolExecutor` | CPU-intensive Aufgaben | `.submit()`, `.result()` |
| Worker-.py-File | Prozesse können Funktionen importieren | `from worker import func` |

#### Die goldene Regel:
> **I/O-bound** (warten auf Netzwerk/Festplatte) → **Threads**
> **CPU-bound** (rechnen, verarbeiten)           → **Prozesse**


In [14]:
print("\n" + "=" * 60)
print("Lern-Notebook abgeschlossen!")
print("Alle Konzepte wurden live demonstriert.")
print("=" * 60)


Lern-Notebook abgeschlossen!
Alle Konzepte wurden live demonstriert.
